# 🧑 Notebook 4: Human Annotation & Metric–Human Misalignment

**Reads:** `metrics_all_models.csv` and the audited human annotation CSV  
**Writes:** `annotated_all_models.csv`, `misalignment_summary.csv`, `iaa_scores.csv`, `metric_classifier_report.csv`

## Research purpose

This notebook connects automatic caption metrics with **human semantic correctness** and the project's four semantic error categories.

Final major results must use **real audited annotations**. Synthetic labels are disabled by default.

In [ ]:
# CELL 1 — Install dependencies
import subprocess, sys
for pkg in ["pandas","numpy","matplotlib","seaborn","scipy","scikit-learn","tqdm"]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])
print("✅ Dependencies ready.")

In [ ]:
# CELL 2 — Imports & configuration
import sys, warnings, random
from pathlib import Path
from collections import Counter
sys.path.insert(0, str(Path(".")))
import config as cfg
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import cohen_kappa_score, confusion_matrix, classification_report
SEED = cfg.SEED
MODELS = list(cfg.MODELS_TO_RUN)
OUTPUT_DIR = cfg.OUTPUT_DIR
FIGURES_DIR = cfg.FIGURES_DIR
random.seed(SEED); np.random.seed(SEED)
MODEL_CAP_COLS = {
    "blip":"blip_caption","blip2":"blip2_caption",
    "ofa":"ofa_caption","vit_gpt2":"vit_gpt2_caption"
}
CANONICAL_ERROR_TYPES = [
    "correct","attribute_mismatch","relation_mismatch",
    "object_misidentification","object_hallucination"
]
print("✅ Config ready:", MODELS)

In [ ]:
# CELL 3 — Real annotation path
# IMPORTANT: point this to the audited human annotation file.
REAL_ANNOTATION_CSV = Path("data/annotation/manual_labels.csv")

# Development-only escape hatch. Keep FALSE for final major experiments.
ALLOW_SIMULATION = False

USE_REAL = REAL_ANNOTATION_CSV.exists()
print("Annotation mode:", "REAL FILE" if USE_REAL else "MISSING REAL FILE")
print("Path:", REAL_ANNOTATION_CSV)

if not USE_REAL and not ALLOW_SIMULATION:
    raise FileNotFoundError(
        "Real annotation CSV not found. Set REAL_ANNOTATION_CSV to the "
        "audited annotation file. Simulation is disabled for final runs."
    )

In [ ]:
# CELL 4 — Load and validate metrics
metrics_path = OUTPUT_DIR / "metrics_all_models.csv"
if not metrics_path.exists():
    raise FileNotFoundError("Run 03_metric_computation.ipynb first.")
df = pd.read_csv(metrics_path)

required = {"image_id","scenario_category","primary_gt_caption"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")
if df["image_id"].isna().any() or df["image_id"].duplicated().any():
    raise ValueError("metrics_all_models.csv must contain one unique non-null image_id per row.")
print(f"✅ Loaded {len(df):,} metric rows.")

## 📝 Section 1 — Load, normalize and validate human annotations

In [ ]:
# CELL 5 — Load REAL annotations; support wide and long formats
if USE_REAL:
    ann = pd.read_csv(REAL_ANNOTATION_CSV)
    if "image_id" not in ann.columns:
        raise ValueError("Annotation file must contain image_id.")

    ANN_COLS = ["human_correct","error_type","error_token",
                "annotator_1","annotator_2","annotator_3"]

    wide = any(f"{c}_{m}" in ann.columns for c in ANN_COLS for m in MODELS)

    if wide:
        if ann["image_id"].duplicated().any():
            raise ValueError("Wide annotation file must have unique image_id values.")
        ai = ann.set_index("image_id")
        missing_ids = set(df.image_id) - set(ai.index)
        if missing_ids:
            raise ValueError(f"Missing annotations for {len(missing_ids)} metric images.")
        for m in MODELS:
            for c in ANN_COLS:
                src = f"{c}_{m}"
                if src in ai.columns:
                    df[src] = df.image_id.map(ai[src])
    else:
        required_long = {"model","human_correct","error_type"}
        missing = required_long - set(ann.columns)
        if missing:
            raise ValueError(f"Invalid annotation schema; missing {sorted(missing)}.")
        if ann.duplicated(["image_id","model"]).any():
            raise ValueError("Duplicate image_id/model annotation pairs found.")
        for m in MODELS:
            ma = ann[ann.model.astype(str) == str(m)].set_index("image_id")
            missing_ids = set(df.image_id) - set(ma.index)
            if missing_ids:
                raise ValueError(f"{m}: missing annotations for {len(missing_ids)} images.")
            for c in ANN_COLS:
                if c in ma.columns:
                    df[f"{c}_{m}"] = df.image_id.map(ma[c])

    for m in MODELS:
        if f"human_correct_{m}" not in df.columns or f"error_type_{m}" not in df.columns:
            raise ValueError(f"{m}: required human annotation columns are missing.")
else:
    print("⚠️ Development simulation requested.")

In [ ]:
# CELL 6 — Optional simulation ONLY for development
if not USE_REAL:
    if not ALLOW_SIMULATION:
        raise RuntimeError("Simulation is disabled.")
    scenario_rate = {
        "gender_ambiguity":.38,"object_confusion":.35,"context_mismatch":.30,
        "counting_errors":.28,"attribute_errors":.25,"general_baseline":.18
    }
    type_dist = {
        "blip":[.45,.25,.18,.12],"blip2":[.38,.28,.20,.14],
        "ofa":[.40,.30,.18,.12],"vit_gpt2":[.50,.22,.16,.12]
    }
    for mi,m in enumerate(MODELS):
        rng = np.random.RandomState(SEED + mi*100)
        a = [[],[],[]]; hc=[]; et=[]; tok=[]
        for _,r in df.iterrows():
            p = scenario_rate.get(r.get("scenario_category","general_baseline"),.25)
            bleu = r.get(f"bleu4_{m}",.2)
            bleu = .2 if pd.isna(bleu) else float(bleu)
            err = rng.rand() < min(.9,max(.05,p*(1+(.25-bleu)*1.5)))
            labels=[]
            for _ in range(3):
                lab = err
                if rng.rand()<.15: lab=not lab
                labels.append(int(not lab))
            for i in range(3): a[i].append(labels[i])
            correct = int(sum(labels)>=2); hc.append(correct)
            if correct:
                et.append("correct"); tok.append(None)
            else:
                e = rng.choice(CANONICAL_ERROR_TYPES[1:],p=type_dist.get(m,[.25]*4))
                et.append(e); tok.append("[SYNTHETIC]")
        for i in range(3): df[f"annotator_{i+1}_{m}"]=a[i]
        df[f"human_correct_{m}"]=hc; df[f"error_type_{m}"]=et; df[f"error_token_{m}"]=tok
    print("⚠️ Synthetic annotations generated for development only.")

In [ ]:
# CELL 7 — Normalize and validate semantic taxonomy
ALIASES = {
    "0":"correct","1":"attribute_mismatch","2":"relation_mismatch",
    "3":"object_misidentification","4":"object_hallucination",
    "correct":"correct","attribute mismatch":"attribute_mismatch",
    "attribute_mismatch":"attribute_mismatch","relation mismatch":"relation_mismatch",
    "relation_mismatch":"relation_mismatch","object misidentification":"object_misidentification",
    "object_misidentification":"object_misidentification",
    "object hallucination":"object_hallucination",
    "object_hallucination":"object_hallucination"
}
def norm_error(v, h):
    if pd.isna(v) or str(v).strip()=="":
        return "correct" if int(h)==1 else np.nan
    key=str(v).strip().lower().replace("-","_")
    if key not in ALIASES: raise ValueError(f"Unknown error_type: {v!r}")
    return ALIASES[key]

for m in MODELS:
    hc, et = f"human_correct_{m}", f"error_type_{m}"
    df[hc]=pd.to_numeric(df[hc],errors="coerce")
    if df[hc].isna().any() or not df[hc].isin([0,1]).all():
        raise ValueError(f"{hc} must contain only 0/1 with no missing values.")
    df[et]=[norm_error(e,h) for e,h in zip(df[et],df[hc])]
    if not df[et].isin(CANONICAL_ERROR_TYPES).all():
        raise ValueError(f"{et} contains invalid taxonomy values.")
    bad=((df[hc]==1)&(df[et]!="correct"))|((df[hc]==0)&(df[et]=="correct"))
    if bad.any(): raise ValueError(f"{m}: human_correct/error_type contradiction in {bad.sum()} rows.")
print("✅ Human annotation taxonomy validated.")

## 📊 Section 2 — Inter-Annotator Agreement

In [ ]:
# CELL 8 — Cohen's kappa + nominal Krippendorff alpha
def krippendorff_alpha_nominal(data):
    arr=np.asarray(data,dtype=object)
    disagree=pairs=0
    for j in range(arr.shape[1]):
        vals=[arr[i,j] for i in range(arr.shape[0]) if not pd.isna(arr[i,j])]
        for i in range(len(vals)):
            for k in range(i+1,len(vals)):
                pairs+=1; disagree+=int(vals[i]!=vals[k])
    Do=disagree/pairs if pairs else 0.0
    vals=[arr[i,j] for i in range(arr.shape[0]) for j in range(arr.shape[1]) if not pd.isna(arr[i,j])]
    if len(vals)<2: return np.nan
    c=Counter(vals); n=len(vals); De=1-sum((v/n)**2 for v in c.values())
    return 1.0 if De==0 else 1-Do/De

rows=[]
for m in MODELS:
    cols=[f"annotator_{i}_{m}" for i in range(1,4) if f"annotator_{i}_{m}" in df.columns]
    if len(cols)<2:
        print(f"{m}: fewer than 2 annotators; IAA skipped."); continue
    arr=[pd.to_numeric(df[c],errors="coerce").to_numpy() for c in cols]
    row={"model":m}
    for i in range(len(arr)):
        for j in range(i+1,len(arr)):
            mask=~np.isnan(arr[i])&~np.isnan(arr[j])
            row[f"kappa_{i+1}_{j+1}"]=cohen_kappa_score(arr[i][mask].astype(int),arr[j][mask].astype(int)) if mask.sum()>1 else np.nan
    row["krippendorff_alpha"]=krippendorff_alpha_nominal(arr)
    rows.append(row)
iaa_df=pd.DataFrame(rows)
iaa_df.to_csv(OUTPUT_DIR/"iaa_scores.csv",index=False)
display(iaa_df)

## 📈 Section 3 — Human Error Distribution

In [ ]:
# CELL 9 — Error type distribution
FIGURES_DIR.mkdir(parents=True,exist_ok=True)
fig,axes=plt.subplots(1,len(MODELS),figsize=(5*len(MODELS),5),squeeze=False)
axes=axes.ravel()
for ax,m in zip(axes,MODELS):
    counts=df[f"error_type_{m}"].value_counts().reindex(CANONICAL_ERROR_TYPES,fill_value=0)
    bars=ax.bar(counts.index,counts.values,edgecolor="white")
    for b,v in zip(bars,counts.values): ax.text(b.get_x()+b.get_width()/2,b.get_height()+.3,str(v),ha="center",fontsize=8)
    ax.set_title(cfg.MODEL_DISPLAY.get(m,m)); ax.set_ylabel("Images"); ax.tick_params(axis="x",rotation=45); ax.grid(axis="y",alpha=.3)
plt.tight_layout(); fig.savefig(FIGURES_DIR/"error_type_distribution.png",dpi=150,bbox_inches="tight"); plt.show()

## ⚡ Section 4 — Metric–Human Misalignment

In [ ]:
# CELL 10 — BLEU-4 binary verdict vs human semantic correctness
BLEU_THRESHOLD=0.20
rows=[]
for m in MODELS:
    bc,hc=f"bleu4_{m}",f"human_correct_{m}"
    if bc not in df.columns: raise ValueError(f"Missing {bc}; rerun Notebook 3.")
    metric=(df[bc]>=BLEU_THRESHOLD).astype(int); human=df[hc].astype(int)
    df[f"metric_correct_{m}"]=metric
    fh=int(((metric==1)&(human==0)).sum()); fl=int(((metric==0)&(human==1)).sum())
    ag=int((metric==human).sum()); n=len(df)
    rows.append({"model":cfg.MODEL_DISPLAY.get(m,m),"n_total":n,
                 "false_high":fh,"false_high_pct":100*fh/n,
                 "false_low":fl,"false_low_pct":100*fl/n,
                 "misalign_pct":100*(fh+fl)/n,"agreement_pct":100*ag/n})
misalign_df=pd.DataFrame(rows)
misalign_df.to_csv(OUTPUT_DIR/"misalignment_summary.csv",index=False)
display(misalign_df)

In [ ]:
# CELL 11 — Confusion matrices
fig,axes=plt.subplots(1,len(MODELS),figsize=(5*len(MODELS),5),squeeze=False); axes=axes.ravel()
for ax,m in zip(axes,MODELS):
    cm=confusion_matrix(df[f"human_correct_{m}"].astype(int),df[f"metric_correct_{m}"].astype(int),labels=[0,1])
    pct=cm/cm.sum()*100
    ann=np.array([[f"{cm[i,j]}\n({pct[i,j]:.1f}%)" for j in range(2)] for i in range(2)])
    sns.heatmap(cm,ax=ax,annot=ann,fmt="",cmap="Blues",square=True,cbar=False,
                xticklabels=["Metric: Incorrect","Metric: Correct"],yticklabels=["Human: Incorrect","Human: Correct"])
    ax.set_title(cfg.MODEL_DISPLAY.get(m,m))
plt.tight_layout(); fig.savefig(FIGURES_DIR/"confusion_matrix_metric_vs_human.png",dpi=150,bbox_inches="tight"); plt.show()

In [ ]:
# CELL 12 — Misalignment bar chart
fig,ax=plt.subplots(figsize=(10,5)); x=np.arange(len(misalign_df)); w=.35
ax.bar(x-w/2,misalign_df.false_high_pct,w,label="False High",alpha=.85)
ax.bar(x+w/2,misalign_df.false_low_pct,w,label="False Low",alpha=.85)
ax.set_xticks(x); ax.set_xticklabels(misalign_df.model); ax.set_ylabel("Images (%)")
ax.set_title("BLEU-4 / Human Semantic Misalignment"); ax.legend(); ax.grid(axis="y",alpha=.3)
plt.tight_layout(); fig.savefig(FIGURES_DIR/"misalignment_bar.png",dpi=150,bbox_inches="tight"); plt.show()

In [ ]:
# CELL 13 — Saved classifier report
classifier=[]
for m in MODELS:
    y=df[f"human_correct_{m}"].astype(int); p=df[f"metric_correct_{m}"].astype(int)
    r=classification_report(y,p,target_names=["Incorrect(0)","Correct(1)"],output_dict=True,zero_division=0)
    classifier.append({
        "model":cfg.MODEL_DISPLAY.get(m,m),"accuracy":r["accuracy"],
        "precision_correct":r["Correct(1)"]["precision"],"recall_correct":r["Correct(1)"]["recall"],
        "f1_correct":r["Correct(1)"]["f1-score"],
        "precision_incorrect":r["Incorrect(0)"]["precision"],"recall_incorrect":r["Incorrect(0)"]["recall"],
        "f1_incorrect":r["Incorrect(0)"]["f1-score"]})
classifier_df=pd.DataFrame(classifier)
classifier_df.to_csv(OUTPUT_DIR/"metric_classifier_report.csv",index=False)
display(classifier_df)

## 🔎 Section 5 — Qualitative Misalignment Examples

In [ ]:
# CELL 14 — Examples for every model
for m in MODELS:
    mc,hc=f"metric_correct_{m}",f"human_correct_{m}"
    cap=MODEL_CAP_COLS[m]; et=f"error_type_{m}"
    print("\n"+"="*80)
    print(cfg.MODEL_DISPLAY.get(m,m),"— FALSE HIGH")
    for _,r in df[(df[mc]==1)&(df[hc]==0)].head(5).iterrows():
        print(f"ID={r.image_id} | BLEU4={r[f'bleu4_{m}']:.3f} | Error={r[et]}")
        print("GT:",str(r.primary_gt_caption)[:120]); print("CAP:",str(r.get(cap,""))[:120])
    print("\n",cfg.MODEL_DISPLAY.get(m,m),"— FALSE LOW")
    for _,r in df[(df[mc]==0)&(df[hc]==1)].head(5).iterrows():
        print(f"ID={r.image_id} | BLEU4={r[f'bleu4_{m}']:.3f} | Human=CORRECT")
        print("GT:",str(r.primary_gt_caption)[:120]); print("CAP:",str(r.get(cap,""))[:120])

In [ ]:
# CELL 15 — Final validation and save
for m in MODELS:
    for c in [f"human_correct_{m}",f"error_type_{m}"]:
        if df[c].isna().any(): raise ValueError(f"{c} contains missing values.")
    if not df[f"human_correct_{m}"].isin([0,1]).all(): raise ValueError(f"{m}: invalid human_correct.")
    if not df[f"error_type_{m}"].isin(CANONICAL_ERROR_TYPES).all(): raise ValueError(f"{m}: invalid error taxonomy.")

df["annotation_source"]="real" if USE_REAL else "simulation"
if not USE_REAL:
    warnings.warn("Synthetic annotations are present. Do not use this output for final major claims.")

out=OUTPUT_DIR/"annotated_all_models.csv"
df.to_csv(out,index=False)
print(f"💾 Saved: {out}")
print(f"Rows: {len(df):,} | Annotation source: {df.annotation_source.iloc[0].upper()}")
print("✅ Notebook 4 COMPLETE.")